In [1]:
pip install mlflow kubernetes

Looking in indexes: https://packages.redhat.com/api/pypi/public-rhai/rhoai/3.5/cpu-ubi9/simple/
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!export MLFLOW_WEBHOOK_ALLOW_PRIVATE_IPS=true
!export MLFLOW_WEBHOOK_ALLOWED_SCHEMES="http,https"

In [9]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient()

# create or reuse experiment
exp = client.create_experiment("webhook-e2e-test")
mlflow.set_experiment(experiment_id=exp)

# run an experiment and log a simple model
with mlflow.start_run() as run:
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_metric("accuracy", 0.95)
    mlflow.sklearn.log_model(
        sk_model=None,  
        artifact_path="model",
    )
    run_id = run.info.run_id
    print(f"run: {run_id}")

# register the model
model_name = "webhook-e2e-model"
try:
    client.create_registered_model(model_name)
    print(f"registered model: {model_name}")
except Exception:
    print(f"model {model_name} already exists")

mv = client.create_model_version(model_name, f"runs:/{run_id}/model", run_id)
print(f"model version: {mv.version}")

# create the webhook
existing = client.list_webhooks()
if not any(w.name == "promote-candidate-model" for w in existing):
    webhook = client.create_webhook(
        name="promote-candidate-model",
        url="http://mlflow-webhook.agent-pack.svc.cluster.local:8000/mlflow",
        events=["model_version_alias.created"],
        secret="secret",
    )
    print(f"webhook: {webhook.webhook_id} ({webhook.status})")
else:
    print("webhook already exists, skipping")

# set alias to trigger the webhook
client.set_registered_model_alias(model_name, "candidate", mv.version)
print(f"alias 'candidate' set on version {mv.version} -- webhook should fire")

2026/09/11 11:11:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/11 11:11:22 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/09/11 11:11:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: webhook-e2e-model, version 1


run: 5398b99bc1354e438e9b253a6c2242f5
🏃 View run awesome-mare-261 at: https://rh-ai.apps.cluster-crzz2.dyn.redhatworkshops.io/mlflow/#/experiments/2/runs/5398b99bc1354e438e9b253a6c2242f5?workspace=agent-pack
🧪 View experiment at: https://rh-ai.apps.cluster-crzz2.dyn.redhatworkshops.io/mlflow/#/experiments/2?workspace=agent-pack
registered model: webhook-e2e-model
model version: 1
webhook: ba8c5e41-3d77-4259-96fe-f50e96f330d3 (ACTIVE)
alias 'candidate' set on version 1 -- webhook should fire
